# 03 RF load and current budget

This is the notebook that tells us whether a candidate voltage/frequency/capacitance target is physically reasonable.

In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from rot_eo_model.constants import DEFAULTS
from rot_eo_model.rf_lumped import cap_current_peak_A, cap_apparent_power_VA, dielectric_loss_W, z_capacitor, reflected_capacitance_F

In [ ]:
freqs = np.linspace(DEFAULTS.f_min_Hz, DEFAULTS.f_max_Hz, 500)
C = DEFAULTS.capacitance_F
Vpk = DEFAULTS.vpi_diff_peak_V
Ipk = cap_current_peak_A(freqs, C, Vpk)
S = cap_apparent_power_VA(freqs, C, Vpk)

plt.figure()
plt.plot(freqs/1e6, Ipk)
plt.xlabel("Frequency [MHz]")
plt.ylabel("EO capacitor current [Apeak]")
plt.title(f"Current for C={C*1e12:.1f} pF, V={Vpk:.0f} Vpeak")
plt.grid(True)

In [ ]:
summary_freqs = np.array([6e6, 10e6, 30e6, 50e6, 100e6, 102e6])
df = pd.DataFrame({
    "f_MHz": summary_freqs/1e6,
    "Xc_ohm": np.abs(z_capacitor(summary_freqs, C)),
    "Ipeak_A": cap_current_peak_A(summary_freqs, C, Vpk),
    "apparent_power_VA": cap_apparent_power_VA(summary_freqs, C, Vpk),
})
df

In [ ]:
for N in [1, 1.5, 2, 3]:
    print(f"Step-up N={N}: primary sees C≈{reflected_capacitance_F(C, N)*1e12:.1f} pF")

In [ ]:
tan_deltas = [1e-4, 5e-4, 1e-3, 5e-3]
for td in tan_deltas:
    P = dielectric_loss_W(freqs, C, Vpk, td)
    plt.plot(freqs/1e6, P, label=f"tanδ={td:g}")
plt.xlabel("Frequency [MHz]")
plt.ylabel("Dielectric loss estimate [W]")
plt.title("Sensitivity to RF loss tangent")
plt.legend(); plt.grid(True)

Change `C` and `Vpk` aggressively. Every added pF at 100 MHz and hundreds of volts matters.